# SOLiD-PINN v0.10.0 — Independent physics foundation
CPU · Run all · Synthetic, no private data. Joint refinement and local SVD only, not complete independent solver verification or inverse PINN.


In [ ]:
from pathlib import Path
from google.colab import files
import subprocess,sys,os,tempfile,json,hashlib
REV="b41845ef4f8c4d148cbe2ddd1da0b55266b4124f"
repo=Path(tempfile.mkdtemp(prefix="solid-foundation-"))/"repo"
subprocess.run(["git","clone","--no-checkout","https://github.com/theerayutbo/SOLiD-PINN.git",str(repo)],check=True)
subprocess.run(["git","checkout","--detach",REV],cwd=repo,check=True)
assert subprocess.check_output(["git","rev-parse","HEAD"],cwd=repo,text=True).strip()==REV
hashes={"canonical_fv.py":"ef028c3fe1c07dcf2c9071b4bf2192d35fec3dcbb2e4c6adda688bfbb584fe58","test_canonical_fv.py":"75aa8d846dfa11cfa5e1f426d98a3478cefbc3dbe08b94e00b2c6e466eec56d4","foundation_sensitivity.py":"f50eac6d427a8fd63cb8d1040d278dacf3ad8cc82e3d723e519b8910441a4384"}
for n,h in hashes.items(): assert hashlib.sha256((repo/"scripts"/n).read_bytes()).hexdigest()==h
env=dict(os.environ,OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1")
subprocess.run([sys.executable,"-m","pip","install","numpy==2.3.5","scipy==1.17.0","pytest"],check=True)
subprocess.run([sys.executable,"-m","pytest","-q","scripts/test_canonical_fv.py"],cwd=repo,env=env,check=True)


## Run synthetic multi-rate stages and local sensitivity


In [ ]:
import numpy as np
subprocess.run([sys.executable,"scripts/canonical_fv.py"],cwd=repo,env=env,check=True)
d=json.loads((repo/"foundation_result.json").read_text())
rows=d["rows"]
assert len(rows)==12
assert {(tuple(x["mesh"]),x["rate"],x["stage"]) for x in rows}=={(mesh,r,s) for mesh in ((4,12),(8,24)) for r in (20,10,5) for s in ("charge","discharge")}
pairs=[]
for a,b in zip(rows[:6],rows[6:]):
    assert (a["rate"],a["stage"])==(b["rate"],b["stage"])
    t=np.linspace(0,min(a["end_s"],b["end_s"]),101)
    err=1000*(np.interp(t,a["time_s"],a["voltage"])-np.interp(t,b["time_s"],b["voltage"]))
    pairs.append(dict(rate=a["rate"],stage=a["stage"],max_sampled_delta_mv=float(max(abs(err))),capacity_delta_pct=100*abs(a["capacity"]/b["capacity"]-1)))
gate=all(p["max_sampled_delta_mv"]<=1 and p["capacity_delta_pct"]<=.5 for p in pairs) and all(x["termination"]=="voltage" and x["mass_residual_C_m2"]<1e-5 and x["algebraic_residual"]<1e-8 and 0<x["min_theta"]<=x["max_theta"]<1 for x in rows)
d.update(joint_mesh_pairs=pairs,joint_mesh_gate_passed=gate,source_revision=REV,source_hashes=hashes,targeted_tests_passed=4)
if gate:
    subprocess.run([sys.executable,"scripts/foundation_sensitivity.py"],cwd=repo,env=env,check=True)
    d["sensitivity"]=json.loads((repo/"foundation_sensitivity.json").read_text())
print("Joint refinement gate:",gate)


## Export evidence


In [ ]:
raw=(json.dumps(d,sort_keys=True,indent=2,allow_nan=False)+"\n").encode()
sha=hashlib.sha256(raw).hexdigest()
dest=Path("/content")/f"SOLiD-PINN_v0100_colab_{sha[:20]}.json"
dest.write_bytes(raw)
assert dest.read_bytes()==raw
print(dest,sha)
files.download(str(dest))
